In [1]:
import torch
import json
from datasets import Dataset, DatasetDict
from torch.utils.data import DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)
from peft import LoraConfig, get_peft_model, PeftModel
from torch.nn import CrossEntropyLoss

import regex as re  
from typing import Optional
import os
import gc
from tqdm import tqdm

In [2]:
class ManualDataset(torch.utils.data.Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            "id": item["id"],
            "premises": item["premises"],
            "conclusion": item["conclusion"],
            "label": item["label"]
        }

In [3]:
class FOLModel:
    def __init__(
        self,
        model_name: str = "/kaggle/input/models/ductri0981/deepseek-model/transformers/default/1/deepseek_model",
        output_dir: str = "/kaggle/working/fol_model",
        max_length: int = 768,
        local_files_only=True
    ):
        self.model_name = model_name
        self.output_dir = output_dir
        self.max_length = max_length
        
        #Load tokenizer
        print("Loading tokenizer...")
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_name,
            trust_remote_code=True,
            local_files_only=local_files_only
        )

        #Load base model.
        print("Loading model...")
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            dtype=torch.float16,
            device_map={"": 0},
            trust_remote_code=True,
            local_files_only=True
        )
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.model.config.use_cache = False
        self.labels = ["True", "False", "Uncertain"]
        self.label_token_ids = {
            label: self.tokenizer(label, add_special_tokens=False).input_ids[0]
            for label in self.labels
        }

    def predict(self, loader):
        all_results = []
    
        self.model.eval()
        with torch.no_grad():
            for batch in tqdm(loader):
                ids = batch["id"]
                labels = batch["label"]
    
                # ====== tạo input RAW ======
                texts = [
                    f"Premise: {p}\nConclusion: {c}\nAnswer:"
                    for p, c in zip(batch["premises"], batch["conclusion"])
                ]
    
                inputs = self.tokenizer(
                    texts,
                    return_tensors="pt",
                    padding=True,
                    truncation=True,
                    max_length=self.max_length
                ).to(self.model.device)
    
                outputs = self.model(**inputs)
    
                logits = outputs.logits
                last_logits = logits[:, -1, :]  # (B, vocab)
    
                batch_preds = []
    
                for i in range(last_logits.size(0)):
                    scores = {
                        label: last_logits[i, token_id].item()
                        for label, token_id in self.label_token_ids.items()
                    }
    
                    pred = max(scores, key=scores.get)
                    batch_preds.append(pred)
    
                for id_, label, pred in zip(ids, labels, batch_preds):
                    all_results.append({
                        "id": id_,
                        "predict": pred,
                        "label": label
                    })
    
        return all_results

In [4]:
import json
with open("/kaggle/input/datasets/ductri0981/manual-test/manual_test.json", 'r', encoding="utf-8") as f:
    dataset = json.load(f)
dataset = ManualDataset(dataset)
loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=False,
)
model = FOLModel()

Loading tokenizer...
Loading model...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

In [5]:
os.environ["TRANSFORMERS_VERBOSITY"] = "info"

results_predict = model.predict(loader)
count_correct = 0
for result_predict in results_predict:
    if result_predict["label"] == result_predict["predict"]:
        count_correct += 1
accuracy = count_correct / len(results_predict)

result = {
    "accuracy": f"{accuracy:.4f}",
    "data": results_predict
}

with open("result.json", 'w', encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=4)

100%|██████████| 11/11 [00:03<00:00,  3.53it/s]
